In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jagravsinghjs/Scamouflage/refs/heads/main/data/Scamouflage.json"
train = pd.read_json(url)

In [ ]:
X = train['text']
Y = train['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=64,
    stratify=Y
)

In [ ]:
import requests
import pickle

# load vectorizer
vec_url = "https://raw.githubusercontent.com/jagravsinghjs/Scamouflage/main/artifacts/vectorizer.pkl"
vectorizer = pickle.loads(requests.get(vec_url).content)

In [ ]:
print(type(vectorizer))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [ ]:
X_train_tfidf = vectorizer.transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    C=1.0
)

In [ ]:
model.fit(X_train_tfidf, Y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, solver='saga')

In [ ]:
print(model.n_iter_)

[20]


In [ ]:
y_pred = model.predict(X_val_tfidf)
y_proba = model.predict_proba(X_val_tfidf)[:, 1]

In [ ]:
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score

In [ ]:
cm = confusion_matrix(Y_val, y_pred)
precision = precision_score(Y_val, y_pred)
recall = recall_score(Y_val, y_pred)
f1 = f1_score(Y_val, y_pred)
roc_auc = roc_auc_score(Y_val, y_proba)

In [ ]:
print(cm,precision,recall,f1,roc_auc)

[[185   5]
 [  5 255]] 0.9807692307692307 0.9807692307692307 0.9807692307692307 0.9979554655870445


In [ ]:
print(X_train_tfidf.shape)

(1798, 4054)


In [ ]:
val_text = X_val.reset_index(drop=True)

for i in range(len(y_pred)):
    if y_pred[i] == 1 and Y_val.reset_index(drop=True)[i] == 0:
        print(val_text[i])
        print("----")

Please review quarterly budget summary.
----
New login detected from your usual device.
----
Your package requires signature upon arrival.
----
Workshop materials shared via drive link.
----
Package returned as requested.
----


In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    vectorizer.transform(X),
    Y,
    cv=5,
    scoring='f1'
)
print(scores)
print(scores.mean())

[0.97120921 0.97297297 0.9787234  0.98084291 0.98062016]
0.9768737314392538


In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

top_positive = coefs.argsort()[-20:]
top_negative = coefs.argsort()[:20]

print("Top scam indicators:")
for i in reversed(top_positive):
    print(feature_names[i])

print("\nTop ham indicators:")
for i in top_negative:
    print(feature_names[i])

Top scam indicators:
gd
ly
our
immediately
now
requires
crypto
security
or
verify
details
click
pay
secure
cancel
dear
here
reward
pending
alert

Top ham indicators:
successfully
the
pm
has been
confirmed
reminder
processed
for
appointment
tomorrow
hi
am
next
been
completed
has
30
thanks
thank you
thank


In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>